# Pull data

In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3

## Functions

In [2]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name='dustin-payment-analysis'):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

## Constants

In [3]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii


## Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

## Read query

In [5]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('SELECT \n'
 '\tsp.bigAccountId,\n'
 '\tsp.FundingMonth,\n'
 '\tDATEADD(MONTH, 24, sp.FundingMonth) AS FundingMonthPlus24,\n'
 '\tsp.MonthOnBooks,\n'
 '\tsp.AccountChargeOff,\n'
 '\tsp.monthOfDefault,\n'
 '\tsp.NetChargeOff,\n'
 '\tsp.RunningNetLoss,\n'
 '\tDATEADD(DAY, 60, c.dtmDue) AS FirstDate60plus,\n'
 '\tCASE \n'
 '\t\tWHEN RunningNetLoss > 0 AND DATEADD(DAY, 60, c.dtmDue) < DATEADD(MONTH, '
 '24, sp.FundingMonth)\n'
 '\t\tTHEN 1\n'
 '\t\tELSE 0\n'
 '\tEND AS bitTarget24Months\n'
 'FROM \n'
 '\telectra.riskdb.dbo.tblReportCOStaticPools_StaticPool sp LEFT OUTER JOIN\n'
 '\t(\n'
 '\t\tSELECT\n'
 '\t\t\tatt.bigAccountId,\n'
 '\t\t\tMIN(c.dtmDue) AS dtmDue\n'
 '\t\tFROM \n'
 '\t\t\telectra.pfsdb.dbo.tblAccountTerms att INNER JOIN '
 'electra.pfsdb.dbo.tblCharges c\n'
 '\t\t\tON att.bigAccountTermId = c.bigAccountTermId\n'
 '\t\tWHERE\n'
 '\t\t\tatt.bigAccountTermTypeId = 1\n'
 '\t\t\tAND c.bigChargeTypeId = 1\n'
 '\t\t\tAND c.bitExtended = 0\n'
 '\t\t\tAND c.bitInvalid = 0\n'
 '\t\t

## Write into df

In [6]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()
# info
print(f'Data contains {df.shape[0]} rows and {df.shape[1]} columns')

Data contains 352765 rows and 10 columns
Wall time: 1min 19s


In [7]:
# preview
df.head()

,bigAccountId,FundingMonth,FundingMonthPlus24,MonthOnBooks,AccountChargeOff,monthOfDefault,NetChargeOff,RunningNetLoss,FirstDate60plus,bitTarget24Months
0,3,2000-06-01,2002-06-01,24,0,NaN,0.0,0.0,NaT,0
1,6,2003-02-01,2005-02-01,24,0,NaN,0.0,0.0,2007-04-24,0
2,11,2000-06-01,2002-06-01,24,0,NaN,0.0,0.0,NaT,0
3,39,2000-06-01,2002-06-01,24,0,NaN,0.0,0.0,NaT,0
4,58,2002-07-01,2004-07-01,24,0,NaN,0.0,0.0,NaT,0


### Save

In [8]:
%%time

# save
str_filename = 'df_targets.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 9.63 s


## Upload to s3

In [9]:
# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'ad_hoc/03_target_creation/{str_filename}', 
    str_bucket_name=str_project,
)

## Clean-up

In [10]:
os.remove(str_local_path)